# Save country level TMREL data as netCDF files

For each country in `country_location_ids.json`, reads `tmrel_{location_id}.csv` and writes `tmrel_{location_id}.nc`

In [1]:
import os
import json
import pathlib
import pandas as pd
import config
from utils.utils import require_dir

In [2]:
# === Setup ===
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
TMREL_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "TMREL")

# Load look up file
lookup_file = "country_location_ids.json"  # {"Armenia": 33, "Azerbaijan": 34, ...}
lookup_path = os.path.join(DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

In [3]:
def process_country(country_name, location_id, dir_path):
    """Read tmrel_{location_id}.csv, process, save as NetCDF."""
    in_file = os.path.join(dir_path, f"tmrel_{location_id}.csv")

    if not os.path.exists(in_file):
        print(f"  SKIP {country_name} (location_id={location_id}): {in_file} not found")
        return None

    df = pd.read_csv(in_file)
    df = df[df["year_id"] == 2020].drop(columns=["year_id", "location_id"])
    df = df.rename(columns={"meanTempCat": "annual_temperature"})

    draw_cols = [c for c in df.columns if c.startswith("tmrel_")]

    long_df = df.melt(
        id_vars=["annual_temperature"],
        value_vars=draw_cols,
        var_name="draws",
        value_name="value",
    )
    long_df["draws"] = long_df["draws"].str.replace("tmrel_", "", regex=False).astype(int)

    da = (
        long_df
        .set_index(["annual_temperature", "draws"])["value"]
        .to_xarray()
    )

    description = (
        "TMREL: Burkart K, Brauer M, Aravkin A et al. Estimating the "
        "cause-specific relative risks of non-optimal temperature "
        "on daily mortality: a two-part modelling approach applied "
        "to the Global Burden of Disease Study"
    )
    da.attrs["description"] = description
    da.attrs["location_id"] = location_id
    da.attrs["country_name"] = country_name

    out_file = f"tmrel_{location_id}.nc"
    out_path = os.path.join(dir_path, out_file)
    print(f" Saving to {out_path}")
    da.to_netcdf(out_path)
    return out_path

In [4]:
# === Main loop ===
saved = []
skipped = []
for country_name, location_id in country_to_id.items():
    result = process_country(country_name, location_id, TMREL_DIR)
    if result is None:
        skipped.append(country_name)
    else:
        saved.append(country_name)

print(f"\nDone. Saved {len(saved)}, skipped {len(skipped)}.")
if skipped:
    print("Skipped (input CSV not found):")
    for c in skipped:
        print(f"  - {c}")

 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_33.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_34.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_35.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_36.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_37.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_38.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_39.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_40.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_41.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_43.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_44.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_45.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_46.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_47.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmrel_48.nc
 Saving to /glade/work/awells/workflow/GBD23/TMREL/tmre

Just the global TMREL:

In [5]:
# === GLOBAL ===

in_file = os.path.join(TMREL_DIR, "tmrel_1.csv")
df = pd.read_csv(in_file)

df = df[df["year_id"] == 2020].drop(columns=["year_id", "location_id"])
df = df.rename(columns={"meanTempCat": "annual_temperature"})

draw_cols = [c for c in df.columns if c.startswith("tmrel_")]

long_df = df.melt(
    id_vars=["annual_temperature"],
    value_vars=draw_cols,
    var_name="draws",
    value_name="value",
)
long_df["draws"] = long_df["draws"].str.replace("tmrel_", "", regex=False).astype(int)

da = (
    long_df
    .set_index(["annual_temperature", "draws"])["value"]
    .to_xarray()
)

description = (
    "TMREL: Burkart K, Brauer M, Aravkin A et al. Estimating the "
    "cause-specific relative risks of non-optimal temperature "
    "on daily mortality: a two-part modelling approach applied "
    "to the Global Burden of Disease Study"
)
da.attrs["description"] = description
da.attrs["location_id"] = 1
da.attrs["country_name"] = "global"

out_file = "tmrel_1.nc"
out_path = os.path.join(TMREL_DIR, out_file)
da.to_netcdf(out_path)
print(f"Saved {out_path}")

Saved /glade/work/awells/workflow/GBD23/TMREL/tmrel_1.nc
